In [ ]:
# Copyright (c) TorchGeo Contributors. All rights reserved.
# Licensed under the MIT License.


# NAIP Road Segmentation

_Written by: TorchGeo Contributors_


## Introduction

This tutorial demonstrates road segmentation inference with the ``Unet_Weights.NAIP_RGBN_RESNET18_CHESAPEAKERSC`` model. We will load NAIP RGB+NIR imagery, run the pre-trained model, and visualize predicted road masks.


## Environment


In [ ]:
%pip install torchgeo matplotlib

## Imports


In [ ]:
import tarfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from torchgeo.models import Unet_Weights, unet

## Download a NAIP sample


In [ ]:
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
archive = data_dir / 'naip.tar'
url = 'https://raw.githubusercontent.com/microsoft/torchgeo/main/tests/data/satlas/naip.tar'
if not archive.exists():
    urllib.request.urlretrieve(url, archive)
with tarfile.open(archive) as f:
    f.extractall(data_dir)

tile_id = '1234_5678.png'
rgb_path = data_dir / 'naip/m_3808245_se_17_1_20110801/tci' / tile_id
nir_path = data_dir / 'naip/m_3808245_se_17_1_20110801/ir' / tile_id
rgb = torch.from_numpy(np.array(Image.open(rgb_path))).permute(2, 0, 1)
nir = torch.from_numpy(np.array(Image.open(nir_path))).unsqueeze(0)
image = torch.cat((rgb, nir), dim=0).float().unsqueeze(0)
image.shape

## Visualize RGB input


In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(rgb.permute(1, 2, 0))
plt.axis('off')
plt.title('NAIP RGB Tile')
plt.show()

## Load pre-trained model


In [ ]:
weights = Unet_Weights.NAIP_RGBN_RESNET18_CHESAPEAKERSC
model = unet(weights=weights).eval()
weights.meta

## Run road segmentation inference


In [ ]:
preprocess = weights.transforms()
inputs = preprocess(image)
with torch.inference_mode():
    logits = model(inputs)
pred = logits.softmax(dim=1)[0, 1]
mask = pred > 0.5
pred.shape, mask.float().mean().item()

## Visualize road probabilities and binary mask


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(rgb.permute(1, 2, 0))
ax[0].set_title('RGB')
ax[1].imshow(pred, cmap='magma')
ax[1].set_title('Road Probability')
ax[2].imshow(mask, cmap='gray')
ax[2].set_title('Road Mask > 0.5')
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()